# RAG (Embedding Search) — `uni_subject` Annotation

For each test entry, retrieves the top-5 most similar education entries from the
training set **within the same country**, shows their assigned subject codes
as labelled examples in the prompt, then asks the LLM to classify.

**Classification model:** Claude Opus 4.6  
**Embedding model:** LM Studio local — set `EMBEDDING_MODEL_ID` below  
**Requires:** `ANTHROPIC_API_KEY` in `.env`

## 1. Imports & config

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
import os
import re
import time
from pathlib import Path

import anthropic
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from openai import OpenAI
from tqdm import tqdm

from corex_eval import evaluate, load_gold, load_inputs, load_training_data

MODEL_ID           = "claude-opus-4-6"
TEMPERATURE        = 0
MAX_TOKENS         = 32

EMBEDDING_MODEL_ID = "text-embedding-bge-m3"
LMSTUDIO_URL       = "http://localhost:1234/v1"
EMBED_BATCH_SIZE   = 32
K                  = 5

CACHE_DIR        = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)
EMBED_CACHE_FILE = CACHE_DIR / "train_embeddings.npy"

client       = anthropic.Anthropic()
embed_client = OpenAI(base_url=LMSTUDIO_URL, api_key="lm-studio")
print(f"Classification model : {MODEL_ID}")
print(f"Embedding model      : {EMBEDDING_MODEL_ID}")

## 2. Load training data

In [ ]:
train_df = load_training_data(
    task="annotation",
    variable="uni_subject",
    features=["subject_label", "uni_subject"],
)
train_df = train_df.dropna(subset=["subject_label", "uni_subject"]).reset_index(drop=True)
train_df["input_text"] = train_df["subject_label"]

# Merge country_label from gold (not in silver)
_gold_df     = load_gold()
_country_map = dict(zip(_gold_df["case_id"].astype(str), _gold_df["country_label"]))
train_df["country_label"] = train_df["case_id"].astype(str).map(_country_map).fillna("unknown")

valid_codes = sorted(train_df["uni_subject"].dropna().unique().tolist())
codes_str   = "\n".join(f"  {c}" for c in valid_codes)

print(f"{len(train_df)} training rows across {train_df['country_label'].nunique()} countries")
print(f"{len(valid_codes)} unique subject codes")

## 3. Compute / load training embeddings (cached)

In [ ]:
def batch_embed(texts):
    all_embeddings = []
    for i in tqdm(range(0, len(texts), EMBED_BATCH_SIZE), desc="Embedding"):
        batch = texts[i : i + EMBED_BATCH_SIZE]
        resp  = embed_client.embeddings.create(input=batch, model=EMBEDDING_MODEL_ID)
        all_embeddings.extend([e.embedding for e in resp.data])
    return np.array(all_embeddings, dtype=np.float32)

def single_embed(text):
    resp = embed_client.embeddings.create(input=[text], model=EMBEDDING_MODEL_ID)
    return np.array(resp.data[0].embedding, dtype=np.float32)

if EMBED_CACHE_FILE.exists():
    train_embeddings = np.load(EMBED_CACHE_FILE)
    print(f"Loaded {len(train_embeddings)} cached embeddings")
else:
    train_embeddings = batch_embed(train_df["input_text"].tolist())
    np.save(EMBED_CACHE_FILE, train_embeddings)
    print(f"Computed and cached {len(train_embeddings)} embeddings")

## 4. Build system prompt

In [ ]:
SYSTEM_PROMPT = (
    "You are an expert political scientist coding university education data "
    "using the CoREx codebook.\n\n"
    "Task: given a subject description, assign the single most appropriate subject code. "
    "You will also be shown similar entries from the training data and their assigned codes "
    "as a reference, proposed by embedding search.\n"
    "These entries are annotated by expert coders — focus on whether the retrieved examples "
    "are a good match; when in doubt rely on your own judgement.\n"
    'Respond with ONLY the exact code label as listed below — nothing else.\n\n'
    f"Valid subject codes:\n{codes_str}"
)
print(f"System prompt: {len(SYSTEM_PROMPT)} chars")

## 5. Load test inputs

In [ ]:
test_df = load_inputs(task="annotation", variable="uni_subject")
test_df["country_label"] = test_df["case_id"].astype(str).map(_country_map).fillna("unknown")
print(f"{len(test_df)} test rows")
test_df[["case_id", "spell_index", "subject_label", "country_label"]].head()

## 6. Retrieval helper

In [ ]:
def find_top_k(query_emb, country, k=K):
    mask = train_df["country_label"].astype(str) == str(country)
    if mask.sum() == 0:
        mask = pd.Series([True] * len(train_df))  # fallback: all countries
    country_embs = train_embeddings[mask]
    country_rows = train_df[mask].reset_index(drop=True)
    norms = np.linalg.norm(country_embs, axis=1)
    sims  = (country_embs @ query_emb) / (norms * np.linalg.norm(query_emb) + 1e-10)
    top_k = np.argsort(sims)[-k:][::-1]
    return [
        (country_rows.iloc[i]["subject_label"],
         country_rows.iloc[i]["uni_subject"])
        for i in top_k
    ]

def format_examples(examples):
    if not examples:
        return ""
    lines = ["\nSimilar entries from the training data:"]
    for i, (subj, code) in enumerate(examples, 1):
        lines.append(f'  {i}. "{subj}" → {code}')
    lines.append("\nNow classify:")
    return "\n".join(lines)

## 7. Inference

In [ ]:
def parse_prediction(text, valid_codes):
    import re
    text = text.strip().strip("\"'")
    if text in valid_codes:
        return text
    # Build map: leading digits → full code
    code_map = {}
    for c in valid_codes:
        m = re.match(r'^(\d+)', c)
        if m:
            code_map[m.group(1)] = c
    # Extract leading digits from prediction
    m = re.match(r'^(\d+)', text)
    if m and m.group(1) in code_map:
        return code_map[m.group(1)]
    # Fallback: substring match on description
    text_lower = text.lower()
    for code in valid_codes:
        parts = re.split(r'\s*[\u2013=]\s*', code, maxsplit=1)
        desc = parts[1].lower() if len(parts) > 1 else code.lower()
        if desc in text_lower or text_lower in desc:
            return code
    return text


predictions = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inference"):
    subject = str(row["subject_label"] or "")
    country = row.get("country_label", "")

    query_emb = single_embed(subject)
    examples  = find_top_k(query_emb, country)
    ex_block  = format_examples(examples)

    user_msg = f"Subject description: {subject}{ex_block}"

    try:
        resp = client.messages.create(
            model=MODEL_ID,
            system=SYSTEM_PROMPT,
            messages=[{"role": "user", "content": user_msg}],
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS,
        )
        raw = resp.content[0].text
    except Exception as e:
        print(f"Error on {row['case_id']}/{row['spell_index']}: {e}")
        raw = ""
        time.sleep(0.5)

    predictions.append(parse_prediction(raw, valid_codes))

predictions_df = test_df[["case_id", "spell_index"]].copy()
predictions_df["predicted_code"] = predictions
predictions_df.head()

## 8. Evaluate

In [ ]:
results = evaluate(
    predictions_df,
    task="annotation",
    variable="uni_subject",
    submit=True,
    experiment_path="/home/tom/projects/corex_eval/experiments/annotation/education/rag_embeddings_uni_subject/config.yaml",
)
pd.Series({k: v for k, v in results.items() if k not in ("per_class", "per_country")})

## 9. Confusion matrix

In [ ]:
from corex_eval.silver import load_silver, get_silver_inputs

plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"]   = "white"

silver_df     = load_silver()
test_case_ids = set(test_df["case_id"].astype(str))
silver_inputs = get_silver_inputs(silver_df, "uni_subject", test_case_ids)

# Normalize both to leading numeric prefix for display
def to_prefix(code):
    m = re.match(r'^(\d+)', str(code))
    return m.group(1) if m else str(code)

silver_inputs["gold_prefix"] = silver_inputs["uni_subject"].map(to_prefix)

merged_cm = predictions_df.merge(
    silver_inputs[["case_id", "spell_index", "gold_prefix", "uni_subject"]],
    on=["case_id", "spell_index"],
    how="inner",
).dropna(subset=["gold_prefix"])

merged_cm["pred_prefix"] = merged_cm["predicted_code"].map(to_prefix)

# Build sorted label list
all_prefixes = sorted(
    set(merged_cm["gold_prefix"]) | set(merged_cm["pred_prefix"]),
    key=lambda x: int(x) if x.isdigit() else 999
)

# Short label: prefix only (codes can be long)
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(merged_cm["gold_prefix"], merged_cm["pred_prefix"],
                      labels=all_prefixes, normalize="true")

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Row-normalised rate")

ax.set_xticks(range(len(all_prefixes)))
ax.set_xticklabels(all_prefixes, fontsize=9, rotation=45, ha="right", rotation_mode="anchor")
ax.set_yticks(range(len(all_prefixes)))
ax.set_yticklabels(all_prefixes, fontsize=9)
ax.set_xlabel("Predicted code", fontsize=11)
ax.set_ylabel("True code", fontsize=11)
ax.set_title("Confusion matrix — RAG Embeddings (uni_subject)\n"
             "(rows = true label, normalised by row)", fontsize=12, fontweight="bold")

for i in range(len(all_prefixes)):
    for j in range(len(all_prefixes)):
        v = cm[i, j]
        if v > 0.01:
            ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                    fontsize=7, color="white" if v > 0.5 else "black")

plt.tight_layout()
plt.show()

print("\nTop misclassifications (true → predicted, rate > 5%):")
for i, ts in enumerate(all_prefixes):
    for j, ps in enumerate(all_prefixes):
        if i != j and cm[i, j] > 0.05:
            print(f"  {ts} → {ps}: {cm[i,j]:.1%}")